In [ ]:
import pandas as pd
import re
import numpy as np
import ast
df = pd.read_csv("naukari.csv")

## Salary Processing Pipeline

1. Read `naukari.csv` into `df`.
2. Define converters:
  - `if not disclosed`
    - then set to "Not Disclosed-Not Disclosed"
  - `convert_dollar_to_lacs(salary)`:
    - strip "$", commas
    - extract numeric values
    - multiply by 80 (INR conversion)
    - format as `low-high` or single value
  - `convert_cr_to_lacs(salary)`:
    - strip "₹"
    - extract numeric values
    - multiply by 1e7
    - format as range or single value
  - `convert_salary_value(x)`:
    - if contains `"Cr"`, call `convert_cr_to_lacs`
    - elif contains `"$"`, call `convert_dollar_to_lacs`
    - else clean up
     - strip commas, quotes, `₹`, `P.A.`, `Lacs`
     - parse hyphen range
     - if numeric and <500, multiply by 100000 (to lacs→annual INR)
     - return `"low-high"` (or same for fixed)
    - non-string: `"Not Disclosed"`

3. Apply to `df["salary"]`:
  - `df["salary"] = df["salary"].apply(convert_salary_value)`

4. Export cleaned values:
  - `df["salary"].to_csv("cleaned_salaries.csv", index=False)`

In [ ]:
#convert salary values to lacs
def convert_dollar_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("$", "").strip()
  salary = salary.replace(",", "")
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 80 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  if result== "":
    print(salary)
  return result
#convert salary values in crores to lacs
def convert_cr_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("\u20B9", "").strip()
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 10000000 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  return result

def convert_salary_value(x):
    if isinstance(x, str) and "Not Disclosed" in x:
        return "0-0"
    if isinstance(x, str) and "Cr" in x:
        return convert_cr_to_lacs(x)
    elif isinstance(x, str) and "$" in x:
        return convert_dollar_to_lacs(x)
    elif isinstance(x, str):
        cleanrange = x.replace(',', '').strip().replace('"', '').strip().replace("₹", "").strip().replace("P.A.", "").strip().replace("Lacs", "").strip().split('(')[0].strip()
        if cleanrange.find('-') != -1:
            parts = cleanrange.split('-')
            if len(parts) == 2:
                try:
                  low = float(parts[0])
                  high = float(parts[1])
                  if low < 500:
                    low *= 100000
                  if high < 500:
                    high *= 100000
                  cleanrange = f"{low}-{high}"
                except ValueError:
                  cleanrange = f"{parts[0]}-{parts[1]}"
            elif len(parts) == 1:
              cleanrange = f"{parts[0]}"
        else:
            try:
              value = float(cleanrange)
              if value < 500:
                value *= 100000
              cleanrange = f"{value}-{value}"
            except ValueError:
              cleanrange = f"{cleanrange}"
        return cleanrange
    else:
        return "0-0"
def clean_experience(x):
    if pd.isna(x) or not isinstance(x, str):
        return x
    x = x.strip()
    if x.lower() in ["not disclosed", "not specified", "not mentioned"]:
        return "Not Disclosed"
    numbers = re.findall(r"\d+\.?\d*", x)
    if len(numbers) == 2:
        return f"{numbers[0]}-{numbers[1]}"
    elif len(numbers) == 1:
        return f"{numbers[0]}-{numbers[0]}"
    else:
        return x

`exp_req` is normalized by `clean_experience`:

- if missing/NaN or not string: kept as-is.
- if `"not disclosed"`/`"not specified"`/`"not mentioned"` (case-insensitive): converted to `"Not Disclosed"`.
- otherwise extract all numeric tokens using regex `\d+\.?\d*`.
- if two numbers found: set `"min-max"` (e.g. `3-5` stays `3-5`).
- if one number found: duplicate as range `"x-x"` (e.g. `4` -> `4-4`).
- if none, leave original text.

Then:
- `df["exp_req"]` set to string type.
- split into two columns:
  - `min_exp` (before `-`)
  - `max_exp` (after `-`)

In [300]:

df["salary"] = df["salary"].apply(convert_salary_value).astype(str)
df[["min_salary", "max_salary"]] = df["salary"].str.split("-", expand=True)
df["min_salary"] = df["min_salary"].astype(float, errors='ignore').fillna(0).astype(float)
df["max_salary"] = df["max_salary"].astype(float, errors='ignore').fillna(0).astype(float)
df["exp_req"] = df["exp_req"].apply(clean_experience).astype(str)
df[["min_exp", "max_exp"]] = df["exp_req"].str.split("-", expand=True)
df["min_exp"] = df["min_exp"].astype(int, errors='ignore').fillna(0).astype(int)
df["max_exp"] = df["max_exp"].astype(int, errors='ignore').fillna(0).astype(int)
df["working_time"] = df["emp_type"].str.split(',').str[0].str.strip()
df["contract_type"] = df["emp_type"].str.split(',').str[1].str.strip()
df["applicants_count"] = df["applicants"].apply(lambda x: re.search(r"\d+", str(x)).group() if re.search(r"\d+", str(x)) else x,)
df["applicants_count"] = df["applicants_count"].astype(int, errors='ignore').fillna(0).astype(int)
df["location_count"] = df["location"].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0).astype(int)
df["key_skills_count"] = df["key_skills"].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0).astype(int)


In [301]:
# import matplotlib.pyplot as plt
# from scipy.stats import norm
# df["rating"]

# # Drop missing values
# data = df["rating"].dropna()

# # Plot histogram (normalized)
# plt.hist(data, bins=30, density=True)

# # Fit normal distribution
# mu, std = norm.fit(data)

# # Create normal curve
# xmin, xmax = plt.xlim()
# x = np.linspace(xmin, xmax, 100)
# y = norm.pdf(x, mu, std)

# # Plot curve
# plt.plot(x, y)

# # Title
# plt.title(f"Rating Distribution (mean={mu:.2f}, std={std:.2f})")

# plt.show()


In [ ]:

#since the rating column mean is exactly falling at 50% and more often in data hence I replace the NaN values with mean of the column to avoid biasing the data with 0 which is not the case in this dataset
meanRating = df["rating"].mean()
#imputation of missing values in rating column with mean of the column to avoid biasing the data with 0 which is not the case in this dataset
df["rating"] = df["rating"].astype(float, errors='ignore').fillna(meanRating).astype(float)

In [ ]:
#implementation for the MLP 
#sorting the columns based on their data types to make it easier for the MLP model to process the data
df = df.reindex(sorted(df.columns, key=lambda x: str(df[x].dtype)), axis=1)

In [ ]:
df_mlp= df[["min_salary", "max_salary", "min_exp", "max_exp", "location_count", "key_skills_count", "applicants_count", "rating"]]